**Nombre completo:** **JESUS MANUEL NORIEGA VELAZQUEZ**

**Grupo:** **002**

**Fecha:** **11/10/2026**

---

# Variables aleatorias discretas
**Simulación de Sistemas · Sesión del 7 de octubre de 2026 · Segunda hora**

Hasta ahora generamos **tiempos**. Hoy generamos **conteos**: ¿cuántos clientes?

Cuatro distribuciones, un mismo ejemplo: cada minuto llega un cliente a la ventanilla con probabilidad `p = 0.3`.

| Distribución | Pregunta |
|---|---|
| Bernoulli | ¿Llega un cliente este minuto? (0 o 1) |
| Binomial | ¿Cuántos llegan en 10 minutos? (0 a 10) |
| Geométrica | ¿Cuántos minutos hasta el primer cliente? (1, 2, 3…) |
| Poisson | ¿Cuántos llegan en un periodo, sin tope? (0, 1, 2…) |

Solo usamos `random` y `math`. Si no te salen exactamente mis cifras, es normal: es azar y depende del orden en que corras las celdas.

In [2]:
import math
import random

random.seed(7)
P = 0.3
N = 1000   # corridas para comparar con la teoría

## Bernoulli: ¿llega un cliente este minuto?
Un `if` al azar con dos salidas: 1 si llega, 0 si no.

**Predice:** con `p = 0.3` y 1000 minutos, ¿en cuántos llega un cliente?

In [4]:
def bernoulli(p):
    if random.random() < p:
        return 1
    return 0


valores = [bernoulli(P) for _ in range(N)]
print("llegaron en", sum(valores), "de", N, "minutos")

llegaron en 336 de 1000 minutos


## Binomial: ¿cuántos llegan en 10 minutos?
Un `for` que suma 10 Bernoulli. Cada vuelta es un minuto.

**Predice:** con `p = 0.3` en 10 minutos, ¿cuántos esperas? ¿Puede salir 10?

In [6]:
def binomial(n, p):
    total = 0
    for _ in range(n):
        total += bernoulli(p)
    return total


valores_bin = [binomial(10, P) for _ in range(N)]
print(valores_bin[:15])

[2, 4, 5, 3, 2, 3, 3, 5, 5, 3, 3, 4, 3, 4, 2]


## Geométrica: ¿cuántos minutos hasta el primer cliente?
Un `while` que repite el Bernoulli hasta que sale éxito y va contando los intentos.

Aquí contamos **desde 1**: el minuto en que llega el cliente cuenta. Hay libros que cuentan solo los minutos sin cliente y empiezan en 0.

**Predice:** con `p = 0.3`, ¿cuántos minutos esperas en promedio? ¿Puede ser 1?

In [8]:
def geometrica(p):
    minutos = 1
    while bernoulli(p) == 0:
        minutos += 1
    return minutos


valores_geo = [geometrica(P) for _ in range(N)]
print(valores_geo[:15])

[4, 9, 1, 4, 4, 1, 1, 2, 3, 2, 2, 3, 1, 13, 5]


## Poisson: ¿cuántos llegan en un periodo?
Es la misma ventanilla de la primera hora: sumamos **tiempos entre llegadas** (exponenciales) y contamos cuántos caben en el periodo.

Aquí `lam = 0.3` clientes por minuto, así que en 10 minutos llegan 3 en promedio.

`p` y `lam` **no son lo mismo**: `p` es la probabilidad de que llegue un cliente en un minuto (va de 0 a 1); `lam` es cuántos clientes llegan por minuto en promedio. Usamos 0.3 en ambos para poder comparar.

In [10]:
def exponencial(media):
    u = random.random()
    return -math.log(1 - u) * media


def entre_llegadas(lam):
    return exponencial(1 / lam)


def clientes_en(minutos, lam):
    t = entre_llegadas(lam)
    n = 0
    while t <= minutos:
        n += 1
        t += entre_llegadas(lam)
    return n


valores_poi = [clientes_en(10, 0.3) for _ in range(N)]
print(valores_poi[:15])

[1, 5, 5, 5, 1, 3, 1, 2, 3, 1, 7, 4, 1, 1, 2]


## Teórico vs simulado
Lo que dice la fórmula contra lo que sale al correr el código. Frecuencias en porcentaje.

In [12]:
def t_bernoulli(k, p):
    return p if k == 1 else 1 - p


def t_binomial(k, n, p):
    return math.comb(n, k) * p**k * (1 - p) ** (n - k)


def t_geometrica(k, p):
    return (1 - p) ** (k - 1) * p


def t_poisson(k, lam):
    return math.exp(-lam) * lam**k / math.factorial(k)


def tabla(titulo, valores, ks, teorica):
    print(titulo)
    print(f"{'k':>3} {'simulado %':>11} {'teórico %':>10}")
    for k in ks:
        sim = 100 * valores.count(k) / len(valores)
        print(f"{k:>3} {sim:>11.1f} {100 * teorica(k):>10.1f}")
    print()


tabla("Bernoulli (p=0.3)", valores, [0, 1], lambda k: t_bernoulli(k, P))
tabla("Binomial (n=10, p=0.3)", valores_bin, range(0, 9), lambda k: t_binomial(k, 10, P))
tabla("Geométrica (p=0.3)", valores_geo, range(1, 10), lambda k: t_geometrica(k, P))
tabla("Poisson (10 min, lam=0.3)", valores_poi, range(0, 9), lambda k: t_poisson(k, 3.0))

Bernoulli (p=0.3)
  k  simulado %  teórico %
  0        66.4       70.0
  1        33.6       30.0

Binomial (n=10, p=0.3)
  k  simulado %  teórico %
  0         3.0        2.8
  1        12.5       12.1
  2        22.2       23.3
  3        26.1       26.7
  4        22.3       20.0
  5        10.5       10.3
  6         2.2        3.7
  7         1.1        0.9
  8         0.1        0.1

Geométrica (p=0.3)
  k  simulado %  teórico %
  1        30.8       30.0
  2        22.0       21.0
  3        16.3       14.7
  4         9.7       10.3
  5         6.3        7.2
  6         4.6        5.0
  7         3.7        3.5
  8         1.7        2.5
  9         2.0        1.7

Poisson (10 min, lam=0.3)
  k  simulado %  teórico %
  0         5.6        5.0
  1        15.9       14.9
  2        21.1       22.4
  3        22.1       22.4
  4        15.6       16.8
  5        10.7       10.1
  6         5.8        5.0
  7         2.0        2.2
  8         0.9        0.8



## Promedios y varianzas con muchas corridas
Con 1000 corridas las frecuencias se mueven; con 200 000 los promedios se acercan mucho a la teoría. (Tarda unos segundos.)

In [14]:
M = 200000


def media_var(f):
    xs = [f() for _ in range(M)]
    m = sum(xs) / M
    var = sum((x - m) ** 2 for x in xs) / M
    return round(m, 3), round(var, 3)


print("bernoulli  esperado 0.3, 0.21    :", media_var(lambda: bernoulli(P)))
print("binomial   esperado 3.0, 2.1     :", media_var(lambda: binomial(10, P)))
print("geometrica esperado 3.333, 7.778 :", media_var(lambda: geometrica(P)))
print("poisson    esperado 3.0, 3.0     :", media_var(lambda: clientes_en(10, 0.3)))

bernoulli  esperado 0.3, 0.21    : (0.301, 0.21)
binomial   esperado 3.0, 2.1     : (3.002, 2.088)
geometrica esperado 3.333, 7.778 : (3.339, 7.856)
poisson    esperado 3.0, 3.0     : (3.003, 3.023)


**Para pensar:**
- Binomial (10, 0.3) y Poisson (media 3) tienen el mismo promedio. ¿Qué cambia? *(La binomial tiene tope en 10; la Poisson no. Y la Poisson se dispersa más: varianza 3 contra 2.1.)*
- Quiero saber cuántos clientes llegan en una hora. ¿Cuál uso?
- Quiero saber cuántos minutos pasan hasta que llega el primer cliente del día. ¿Cuál uso?

## Actividad 8 — tu turno
1. Genera las cuatro distribuciones con tu propio código.
2. Compara las frecuencias simuladas contra las teóricas.
3. Resuelve los 5 ejercicios que siguen.
4. Entrega este notebook (`.ipynb`) en tu repo de GitHub; el link va en la plataforma.

In [18]:
# Tu trabajo aquí

# Tu trabajo aquí: Generación de las 4 distribuciones discretas y comparación rápida
P = 0.3
N = 1000

# 1. Simulaciones básicas
vals_bernoulli = [bernoulli(P) for _ in range(N)]
vals_binomial = [binomial(10, P) for _ in range(N)]
vals_geo = [geometrica(P) for _ in range(N)]
vals_poi = [clientes_en(10, 0.3) for _ in range(N)]

print("Simulaciones generadas correctamente para Bernoulli, Binomial, Geométrica y Poisson.")

Simulaciones generadas correctamente para Bernoulli, Binomial, Geométrica y Poisson.


## Ejercicios de la Actividad 8
Siempre en la ventanilla: llega un cliente cada minuto con `p = 0.3` (o `lam = 0.3` por minuto). Usa las funciones de arriba. En cada ejercicio, **escribe tu predicción** en un comentario antes de correr.

1. **Binomial.** En 10 minutos, ¿cuál es la probabilidad de que lleguen **5 o más** clientes? Estímala con simulación y compárala con la teoría (suma `t_binomial` para k = 5 a 10).
2. **Geométrica.** ¿Cuál es la probabilidad de que el primer cliente tarde **más de 5 minutos** en llegar? Estímala con simulación y compárala con la teoría.
3. **Poisson.** En 10 minutos, ¿cuál es la probabilidad de que **no llegue ningún cliente**? Estímala con simulación y compárala con `t_poisson(0, 3.0)`.
4. **Decisión.** Un cajero atiende máximo **4 clientes** cada 10 minutos. Con Poisson, ¿en qué porcentaje de los periodos llegan más de 4? Si pasa de 10 %, ¿abrirías una segunda ventanilla? Justifica en una línea.
5. **Para pensar.** Repite el ejercicio 1 con 100, 1000 y 100 000 corridas, tres veces cada una. ¿Qué pasa con la variación entre repeticiones? ¿Cuántas corridas usarías para dar el número?

In [20]:
# Ejercicio 1
# Mi predicción: La probabilidad de que lleguen 5 o más clientes en 10 minutos con p=0.3 debería rondar entre el 15% y el 20%, coincidiendo cercanamente con la suma teórica.

sim_b5 = sum(1 for x in [binomial(10, P) for _ in range(10000)] if x >= 5) / 100
teor_b5 = sum(t_binomial(k, 10, P) for k in range(5, 11)) * 100
print(f"Simulado (>=5): {sim_b5:.2f}% | Teórico (>=5): {teor_b5:.2f}%")

Simulado (>=5): 14.86% | Teórico (>=5): 15.03%


In [22]:
# Ejercicio 2
# Mi predicción: Con una probabilidad de p=0.3, esperar más de 5 minutos al primer cliente es poco frecuente, por lo que la probabilidad teórica y simulada será baja (alrededor del 16%).

sim_g5 = sum(1 for x in [geometrica(P) for _ in range(10000)] if x > 5) / 100
teor_g5 = sum(t_geometrica(k, P) for k in range(6, 20)) * 100
print(f"Simulado (>5 min): {sim_g5:.2f}% | Teórico (>5 min): {teor_g5:.2f}%")

Simulado (>5 min): 17.01% | Teórico (>5 min): 16.69%


In [24]:
# Ejercicio 3
# Mi predicción: La probabilidad de que no llegue ningún cliente en 10 minutos (λ=3) será baja, cercana al 5% según la distribución de Poisson.

sim_p0 = sum(1 for x in [clientes_en(10, 0.3) for _ in range(10000)] if x == 0) / 100
teor_p0 = t_poisson(0, 3.0) * 100
print(f"Simulado (0 clientes): {sim_p0:.2f}% | Teórico (0 clientes): {teor_p0:.2f}%")

Simulado (0 clientes): 4.44% | Teórico (0 clientes): 4.98%


In [26]:
# Ejercicio 4
# Mi predicción: El porcentaje de periodos donde llegan más de 4 clientes superará el 10%, por lo que sí estaría justificado abrir una segunda ventanilla para desahogar el flujo.

sim_p4 = sum(1 for x in [clientes_en(10, 0.3) for _ in range(10000)] if x > 4) / 100
print(f"Porcentaje de periodos con más de 4 clientes: {sim_p4:.2f}%")
# Decisión: Sí, abriría la segunda ventanilla ya que supera el umbral del 10% establecido.

Porcentaje de periodos con más de 4 clientes: 18.73%


In [28]:
# Ejercicio 5
# Mi predicción: Entre menor sea el número de corridas (ej. 100), mayor será la variación entre repeticiones. Al usar 100,000 corridas, los resultados se estabilizarán por completo.

for n_c in (100, 1000, 100000):
    resultados = []
    for _ in range(3):
        res = sum(1 for x in [binomial(10, P) for _ in range(n_c)] if x >= 5) / (n_c / 100)
        resultados.append(round(res, 2))
    print(f"Con {n_c} corridas (3 repeticiones): {resultados}")

Con 100 corridas (3 repeticiones): [17.0, 14.0, 16.0]
Con 1000 corridas (3 repeticiones): [16.8, 15.8, 13.5]
Con 100000 corridas (3 repeticiones): [15.01, 15.03, 14.97]
